# 📘 Notebook 07: N-gram Language Models

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module C: Counting Words · Notebook 1 of 3 in this module · (7 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **language model** is and name several things it is used for
- Derive the **n-gram** approximation from the chain rule and the Markov assumption
- Estimate n-gram probabilities by **counting**, on a toy corpus and on real novels
- **Generate** text from a model, and describe how the output changes with the length of the context
- Explain the **zero-probability problem** and repair it with **smoothing** and **interpolation**
- Evaluate a language model with **perplexity** on held-out text

> **Prerequisites:** Notebooks 01, 03 and 06. Basic probability: a probability is a number between 0 and 1, and the probabilities of all possible outcomes add up to 1.
>
> 🔭 **Why this notebook matters:** predicting the next word sounds like a party trick. It is the single most important idea in modern NLP. The large language models behind today's AI assistants do exactly one thing, predict the next token, and everything else follows from doing it extremely well. In this notebook you will build the simplest machine of that kind, see it write text, and meet the measure used to judge every language model since.

> ℹ️ **Setup note.** Run this cell once.

In [ ]:
import nltk
for package in ["gutenberg", "punkt", "punkt_tab"]:
    nltk.download(package, quiet=True)

import math
import random
from collections import Counter

## 1. What is a language model?

### Intuition first
Finish the sentence: *"I would like a cup of ..."*. You thought of *tea* or *coffee*, and certainly not of *pavement* or *running*. You have a strong sense of which words are likely to come next. Your phone's keyboard has a weaker version of the same sense when it offers three suggestions above the keys.

A **language model** is a system that assigns probabilities to sequences of words. Equivalently, given the words so far, it gives a probability to every possible **next word**.

### What is it good for?
Whenever a system must choose between several candidate sentences, a language model can say which one sounds most like real language:

- **Speech recognition:** *"recognise speech"* or *"wreck a nice beach"*? The sounds are nearly identical. The probabilities are not.
- **Spelling correction:** *"a peace of cake"* contains no misspelt word, yet it is improbable.
- **Machine translation:** pick the most natural of several possible translations.
- **Text generation:** produce text one word at a time by repeatedly choosing a likely next word.

### Formal definition
A language model defines a probability $P(w_1, w_2, \dots, w_n)$ for any sequence of words. By the **chain rule** of probability, this can always be written as a product of next-word predictions:

$$P(w_1, \dots, w_n) = P(w_1)\; P(w_2 \mid w_1)\; P(w_3 \mid w_1, w_2) \cdots P(w_n \mid w_1, \dots, w_{n-1})$$

The difficulty is the later factors. To estimate $P(w_{20} \mid w_1 \dots w_{19})$ by counting, we would need to have seen that exact 19-word beginning many times. We almost never have.

## 2. The n-gram approximation

### Intuition first
To guess the next word of *"...and after the long walk she asked for a cup of"*, you hardly need the beginning of the sentence. The last two or three words do most of the work. So let us simply pretend that **only the last few words matter**.

### Formal definition
The **Markov assumption** (the same one as in Notebook 06) says that the next word depends only on the previous $n-1$ words. A sequence of $n$ consecutive words is called an **n-gram**:

| Model | Context used | Example of what is estimated |
|---|---|---|
| **unigram** ($n=1$) | none | $P(\text{tea})$ |
| **bigram** ($n=2$) | 1 previous word | $P(\text{tea} \mid \text{of})$ |
| **trigram** ($n=3$) | 2 previous words | $P(\text{tea} \mid \text{cup of})$ |

For a bigram model the probabilities are estimated by counting and dividing:

$$P(w_i \mid w_{i-1}) = \frac{\text{count}(w_{i-1}\, w_i)}{\text{count}(w_{i-1})}$$

*Of all the times we saw the previous word, how often was it followed by this one?*

Let us do it by hand on a corpus of three sentences. We add a marker `<s>` at the start of each sentence and `</s>` at the end, so that the model also learns how sentences begin and end.

In [ ]:
tiny_corpus = [
    "i like green tea",
    "i like black coffee",
    "you like green apples",
]

pair_counts = Counter()
first_word_counts = Counter()

for sentence in tiny_corpus:
    tokens = ["<s>"] + sentence.split() + ["</s>"]
    for first, second in zip(tokens, tokens[1:]):
        pair_counts[(first, second)] += 1
        first_word_counts[first] += 1

def bigram_probability(first, second):
    return pair_counts[(first, second)] / first_word_counts[first]

print("P(i | <s>)      =", round(bigram_probability("<s>", "i"), 3))
print("P(like | i)     =", round(bigram_probability("i", "like"), 3))
print("P(green | like) =", round(bigram_probability("like", "green"), 3))
print("P(tea | green)  =", round(bigram_probability("green", "tea"), 3))

Two of the three sentences start with *i*, so $P(\text{i} \mid \text{<s>}) = 2/3$. The word *like* is followed by *green* in two cases out of three. And so on.

The probability of a whole sentence is the product of its bigram probabilities.

In [ ]:
def sentence_probability(sentence):
    tokens = ["<s>"] + sentence.split() + ["</s>"]
    probability = 1.0
    for first, second in zip(tokens, tokens[1:]):
        probability *= bigram_probability(first, second)
    return probability

for sentence in ["i like green tea", "i like green apples", "you like black coffee", "i like tea"]:
    print(f"{sentence:<24} {sentence_probability(sentence):.3f}")

Look at the second and third lines. *"i like green apples"* and *"you like black coffee"* are **not in the corpus**, yet the model gives them a sensible probability. By breaking sentences into small overlapping pieces, the model has learned to recombine them. This is the model **generalising**, and it is the whole point.

The last line shows the price. *"i like tea"* is a perfectly good sentence, and the model says it is impossible, because the pair *like tea* never occurred. Keep that zero in mind. We will return to it in Section 5.

## 3. Training on real text

Three sentences cannot teach us much. Let us train on three novels by Jane Austen, about 370,000 words. We lower-case the text, keep only the words, and hold out one sentence in ten for testing.

In [ ]:
from nltk.corpus import gutenberg

def load_sentences(file_id):
    sentences = []
    for sentence in gutenberg.sents(file_id):
        words = [word.lower() for word in sentence if word.isalpha()]
        if len(words) >= 3:
            sentences.append(words)
    return sentences

sentences = []
for file_id in ["austen-emma.txt", "austen-persuasion.txt", "austen-sense.txt"]:
    sentences += load_sentences(file_id)

random.seed(0)
random.shuffle(sentences)
cut = int(0.9 * len(sentences))
train, test = sentences[:cut], sentences[cut:]

print(f"Training sentences: {len(train):,}")
print(f"Test sentences:     {len(test):,}")
print(f"Training words:     {sum(len(sentence) for sentence in train):,}")
print(train[0])

One function is enough to train a model for any $n$. For each position in each sentence it records which word followed which **context**, where the context is the tuple of the previous $n-1$ words.

In [ ]:
def train_model(sentences, n):
    model = {}                                         # model[context] is a Counter of next words
    for sentence in sentences:
        tokens = ["<s>"] * (n - 1) + sentence + ["</s>"]
        for i in range(n - 1, len(tokens)):
            context = tuple(tokens[i - n + 1:i])
            if context not in model:
                model[context] = Counter()
            model[context][tokens[i]] += 1
    return model

unigram = train_model(train, 1)
bigram = train_model(train, 2)
trigram = train_model(train, 3)

print("After 'she':        ", bigram[("she",)].most_common(6))
print("After 'she was':    ", trigram[("she", "was")].most_common(6))
print("After 'i am':       ", trigram[("i", "am")].most_common(6))

This is the keyboard on your phone. Each list is the model's set of suggestions for the next word. Compare the first two lines: adding one word of context changes the predictions completely, from verbs to the words that typically follow *she was*.

## 4. Generating text

### Intuition first
If the model can score the next word, it can also **write**. Start with the start marker. Look up the possible next words and pick one at random, in proportion to how often each was seen. Add it to the text, slide the context along by one word, and repeat until the end marker comes up.

In [ ]:
def generate(model, n, max_words=25):
    context = ["<s>"] * (n - 1)
    words = []
    while len(words) < max_words:
        counts = model[tuple(context[len(context) - (n - 1):])]
        word = random.choices(list(counts.keys()), weights=list(counts.values()))[0]
        if word == "</s>":
            break
        words.append(word)
        context.append(word)
    return " ".join(words)

fourgram = train_model(train, 4)

random.seed(4)
for name, model, n in [("unigram", unigram, 1), ("bigram", bigram, 2), ("trigram", trigram, 3), ("4-gram", fourgram, 4)]:
    print(f"{name}:")
    for i in range(2):
        print("   ", generate(model, n))

Read the four pairs in order.

- The **unigram** model picks each word independently. The result is a bag of frequent words with no grammar.
- The **bigram** model links each word to the one before. Short stretches are grammatical, but the sentence wanders.
- The **trigram** model produces phrases that sound like Jane Austen.
- The **4-gram** model is the most fluent of all.

Before admiring the 4-gram model, we should ask why it is so good. Let us count how many contexts in each model have **only one** possible continuation.

In [ ]:
for name, model in [("bigram", bigram), ("trigram", trigram), ("4-gram", fourgram)]:
    single = 0
    for context, counts in model.items():
        if len(counts) == 1:
            single += 1
    print(f"{name:<8} {len(model):>8,} contexts, {single / len(model):.0%} with a single possible next word")

In the 4-gram model the great majority of contexts were seen with exactly one continuation. When the model meets such a context it has no choice: it reproduces the original text. Its fluency is largely **memory**, not skill. It is reciting Jane Austen, a few words at a time.

This is a trade-off you will meet in every model in this course. A longer context gives more accurate predictions, and needs vastly more data before those predictions reflect the language and not just the training text.

## 5. The problem of zeros

### Intuition first
In the toy corpus, *"i like tea"* received probability zero. Is that a toy problem? Let us measure it on real data: what share of the n-grams in the **test** sentences never occurred in training?

In [ ]:
def unseen_share(model, n, sentences):
    unseen = 0
    total = 0
    for sentence in sentences:
        tokens = ["<s>"] * (n - 1) + sentence + ["</s>"]
        for i in range(n - 1, len(tokens)):
            context = tuple(tokens[i - n + 1:i])
            if context not in model or tokens[i] not in model[context]:
                unseen += 1
            total += 1
    return unseen / total

for name, model, n in [("unigram", unigram, 1), ("bigram", bigram, 2), ("trigram", trigram, 3), ("4-gram", fourgram, 4)]:
    print(f"{name:<8} {unseen_share(model, n, test):.0%} of test n-grams were never seen in training")

These are new sentences by the **same author** on the same subjects, and still a large share of their n-grams are new. An unsmoothed model gives every one of them probability zero, and therefore gives probability zero to almost every test sentence. Zipf's law from Notebook 01 is the cause: most word combinations are rare, and no corpus is large enough to contain them all.

Not having seen something is weak evidence that it is impossible. The model needs to keep a little probability in reserve for the unseen.

### Smoothing
The simplest remedy is to pretend that every possible n-gram occurred a little more often than it did.

**Add-k smoothing** adds a small constant $k$ to every count:

$$P(w \mid \text{context}) = \frac{\text{count}(\text{context}, w) + k}{\text{count}(\text{context}) + k \cdot V}$$

where $V$ is the size of the vocabulary. The term $k \cdot V$ in the denominator keeps the probabilities adding up to 1. With $k = 1$ this is called **Laplace smoothing**.

In [ ]:
vocabulary = set()
for sentence in train:
    vocabulary.update(sentence)
V = len(vocabulary) + 2          # plus the end marker, plus one slot for words never seen at all

def context_totals(model):
    return {context: sum(counts.values()) for context, counts in model.items()}

EMPTY = Counter()

def smoothed_probability(model, totals, context, word, k):
    count = model.get(context, EMPTY)[word]
    return (count + k) / (totals.get(context, 0) + k * V)

bigram_totals = context_totals(bigram)

for word in ["was", "is", "smiled", "telephoned"]:
    print(f"P({word} | she) = {smoothed_probability(bigram, bigram_totals, ('she',), word, k=0.01):.6f}")

The words that were seen after *she* keep probabilities in proportion to their counts. *she telephoned* never occurs in these novels (the telephone had not been invented), and it now has a small probability that is no longer zero.

## 6. Measuring a language model: perplexity

### Intuition first
How do we decide whether one language model is better than another? Show both some text they have never seen. The better model is the one that is **less surprised**: it gave higher probability to the words that actually came.

**Perplexity** turns that into a single number with a helpful interpretation. A perplexity of 100 means that, on average, the model was as uncertain as if it had to choose between 100 equally likely words at every step. **Lower is better.** A model that knew nothing and spread its probability evenly over the vocabulary would have a perplexity equal to the vocabulary size.

### Formal definition
For a test text of $N$ words,

$$\text{perplexity} = \exp\Big(-\frac{1}{N}\sum_{i=1}^{N} \log P(w_i \mid \text{context}_i)\Big)$$

It is the inverse of the geometric mean of the probabilities the model assigned to the words that occurred.

In [ ]:
def perplexity(model, n, sentences, k):
    totals = context_totals(model)
    log_sum = 0.0
    count = 0
    for sentence in sentences:
        tokens = ["<s>"] * (n - 1) + sentence + ["</s>"]
        for i in range(n - 1, len(tokens)):
            context = tuple(tokens[i - n + 1:i])
            log_sum += math.log(smoothed_probability(model, totals, context, tokens[i], k))
            count += 1
    return math.exp(-log_sum / count)

print(f"Vocabulary size: {V:,}\n")
print(f"{'':<9} {'k = 1':>9} {'k = 0.1':>9} {'k = 0.01':>9}")
for name, model, n in [("unigram", unigram, 1), ("bigram", bigram, 2), ("trigram", trigram, 3)]:
    row = [perplexity(model, n, test, k) for k in [1, 0.1, 0.01]]
    print(f"{name:<9} {row[0]:>9,.0f} {row[1]:>9,.0f} {row[2]:>9,.0f}")

There is a lot to read in this small table.

- Every model is far better than chance: the perplexities are a small fraction of the vocabulary size.
- **The choice of $k$ matters enormously** for the bigram and trigram models. Adding a full count of 1 to every one of thousands of possible next words takes far too much probability away from the words that were really seen.
- With a suitable $k$ the **bigram** model clearly beats the unigram model: context helps.
- The **trigram** model is *worse* than the bigram model. Its contexts are so sparse that add-k smoothing, which treats all unseen words alike, does it more harm than the extra context does good.

That last finding seems to contradict Section 4, where trigrams produced better text. It does not. The trigram model's knowledge is excellent **where it has data** and useless where it has none. We need a way to use it only when it can be trusted.

### Interpolation: combine the models
The idea is simple and very effective. Do not choose between the three models. **Mix** them:

$$P(w \mid w_{i-2}, w_{i-1}) = \lambda_3 \, P_{\text{trigram}} + \lambda_2 \, P_{\text{bigram}} + \lambda_1 \, P_{\text{unigram}}$$

with weights $\lambda$ that add up to 1. When the trigram has seen the context, it contributes its sharp prediction. When it has not, its term is zero and the bigram and unigram terms carry the estimate. Only the unigram needs smoothing.

In [ ]:
unigram_totals = context_totals(unigram)
trigram_totals = context_totals(trigram)

def interpolated_probability(two_before, one_before, word, weights):
    l3, l2, l1 = weights
    p3 = p2 = 0.0
    if (two_before, one_before) in trigram:
        p3 = trigram[(two_before, one_before)][word] / trigram_totals[(two_before, one_before)]
    if (one_before,) in bigram:
        p2 = bigram[(one_before,)][word] / bigram_totals[(one_before,)]
    p1 = smoothed_probability(unigram, unigram_totals, (), word, k=1)
    return l3 * p3 + l2 * p2 + l1 * p1

def interpolated_perplexity(sentences, weights):
    log_sum = 0.0
    count = 0
    for sentence in sentences:
        tokens = ["<s>", "<s>"] + sentence + ["</s>"]
        for i in range(2, len(tokens)):
            log_sum += math.log(interpolated_probability(tokens[i - 2], tokens[i - 1], tokens[i], weights))
            count += 1
    return math.exp(-log_sum / count)

print(f"Interpolated trigram model: {interpolated_perplexity(test, (0.3, 0.5, 0.2)):,.0f}")

The mixture is better than any of the single models in the table. The trigram's knowledge is finally being used, without its blind spots doing damage.

> ⚠️ **Common pitfalls**
>
> - Computing perplexity on the **training** text. A 4-gram model that has memorised its training data scores brilliantly there and terribly on anything new.
> - Comparing perplexities of models with **different vocabularies** or different tokenisation. The numbers are only comparable when both models predict the same sequence of tokens.
> - Choosing $k$ or the $\lambda$ weights by trying values on the **test** set, as we did above for illustration. Properly, such settings are tuned on a third, separate piece of data, and the test set is used once at the end.

## 7. What n-gram models cannot do

N-gram models were the backbone of speech recognition and machine translation for decades, and they are still useful when speed and simplicity matter. Their limits are equally clear.

**They have a short memory.** In *"The keys that I left on the kitchen table next to the bread **are** missing"*, the verb must agree with *keys*, eleven words earlier. No trigram can see that far, and making $n$ larger runs straight into the sparsity problem.

**They cannot share what they learn between similar words.** Having seen *"a cup of tea"* many times tells an n-gram model nothing about *"a mug of tea"*, because *cup* and *mug* are simply different symbols. Each word is an island.

Here is the score for this notebook's answer to the question of the course.

| | Word counts | N-gram model |
|---|---|---|
| Turns text into numbers | yes | yes |
| Uses word order | no | yes, a few words back |
| Knows that two words are similar | no | no |
| Handles long-range dependencies | no | no |

The second limitation is repaired in Module D, where words become vectors and similar words become neighbours. The first is repaired in Module E, by neural networks that carry information across a whole sentence. They are still language models, trained to predict the next word and evaluated with perplexity, exactly as here.

---
## ✏️ Exercises

### Exercise 1 (By hand, then by machine)
Using the toy corpus of Section 2, work out **on paper** the probability of the sentence *"you like green tea"*. Then check your answer with `sentence_probability`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
# P(you | <s>) = 1/3     one of the three sentences starts with "you"
# P(like | you) = 1      "you" is always followed by "like"
# P(green | like) = 2/3
# P(tea | green) = 1/2
# P(</s> | tea) = 1
# product = 1/3 * 1 * 2/3 * 1/2 * 1 = 1/9
print(sentence_probability("you like green tea"), 1 / 9)
```

*This sentence does not appear in the corpus, and the model still finds it plausible.*
</details>

### Exercise 2 (A predictive keyboard)
Write `suggest(text, how_many=3)` that takes the beginning of a sentence and returns the most likely next words. Use the trigram model when the last two words have been seen, and fall back to the bigram model when they have not. Try it on `"i hope you"`, `"it was a"` and `"my dear"`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def suggest(text, how_many=3):
    words = ["<s>", "<s>"] + text.lower().split()
    two_words = (words[-2], words[-1])
    if two_words in trigram:
        counts = trigram[two_words]
    else:
        counts = bigram.get((words[-1],), Counter())
    return [word for word, count in counts.most_common(how_many) if word != "</s>"]

for text in ["i hope you", "it was a", "my dear"]:
    print(f"{text:<12} -> {suggest(text)}")
```

*Falling back to a shorter context when the longer one is unknown is called backoff. It is the same idea as interpolation, in a simpler form.*
</details>

### Exercise 3 (The commonest phrases)
Find the ten most frequent **trigrams** in the training sentences that do not contain a sentence marker. What kind of phrases are they?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
phrase_counts = Counter()
for (first, second), counts in trigram.items():
    if first == "<s>" or second == "<s>":
        continue
    for word, count in counts.items():
        if word != "</s>":
            phrase_counts[(first, second, word)] += count

for phrase, count in phrase_counts.most_common(10):
    print(f"{' '.join(phrase):<22} {count}")
```

*The top of the list is made of grammatical glue and of the fixed phrases of polite conversation. Frequent n-grams reveal the formulas of a style.*
</details>

### Exercise 4 (Find the best k)
For the **bigram** model, compute the perplexity on the test set for each `k` in `[1, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001, 0.0001]`. Which value is best? What happens when `k` becomes very small, and why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for k in [1, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001, 0.0001]:
    print(f"k = {k:<7} perplexity {perplexity(bigram, 2, test, k):>8,.0f}")
```

*Perplexity falls as k shrinks, reaches a minimum, and then rises again. With a large k, too much probability is given away to words never seen in that context. With a tiny k, the unseen pairs that do turn up in the test text are punished very heavily. The best value balances the two.*
</details>

### Exercise 5 (Who wrote this?)
A language model trained on one author should be less surprised by that author than by anyone else. Using the interpolated model, compute the perplexity of (a) the Austen test sentences, (b) *Alice in Wonderland* (`carroll-alice.txt`) and (c) the King James Bible (`bible-kjv.txt`, use only the first 3,000 sentences).

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
weights = (0.3, 0.5, 0.2)
print(f"Austen (unseen sentences): {interpolated_perplexity(test, weights):>7,.0f}")
print(f"Lewis Carroll:             {interpolated_perplexity(load_sentences('carroll-alice.txt'), weights):>7,.0f}")
print(f"King James Bible:          {interpolated_perplexity(load_sentences('bible-kjv.txt')[:3000], weights):>7,.0f}")
```

*The further a text is from the training data in vocabulary and style, the higher the perplexity. This simple effect has practical uses: identifying the language of a text, attributing authorship, and detecting when a system is being asked to handle a kind of text it was never trained on.*
</details>

### Exercise 6 (Inventing words, a little harder)
N-gram models work on any sequence, including the **letters** of a word. Train a model on the letters of all the distinct words in the training vocabulary, using a context of three letters, and generate fifteen new "words". (Hint: `train_model` accepts any list of lists, so turn each word into a list of its letters. Then use `generate` and join the result without spaces.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
letter_lists = [list(word) for word in sorted(vocabulary)]
letter_model = train_model(letter_lists, 4)

random.seed(2)
invented = []
while len(invented) < 15:
    word = generate(letter_model, 4, max_words=14).replace(" ", "")
    if word not in vocabulary and len(word) > 4:
        invented.append(word)
print(invented)
```

*Most of the results are not English words, but they are pronounceable and look English, because the model has learned which letters tend to follow which. Models of letter sequences are how software guesses the language of a text from a single line, and the idea of working below the level of the word returns in Notebook 15.*
</details>

## 🔑 Key takeaways

- A **language model** assigns probabilities to word sequences, or equivalently predicts the next word.
- The **n-gram** model assumes the next word depends only on the previous $n-1$ words, and estimates its probabilities by **counting**.
- Breaking text into overlapping n-grams lets the model **generalise** to sentences it has never seen.
- Longer contexts give more fluent text, and increasingly that fluency is **memorisation** of the training data.
- Most possible n-grams never occur in any corpus. **Smoothing** and **interpolation** reserve probability for the unseen.
- **Perplexity** on held-out text measures how surprised a model is. Lower is better.
- N-gram models have a **short memory** and treat every word as **unrelated** to every other.

---
**Next:** *Notebook 08: Bag of Words, TF-IDF and Search*, where counting is put to a different use: representing whole documents as vectors, so that we can compare them and search through them.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*